# 🚀 Обучение Модели А (Baseline ResNet) — Google Colab

Этот блокнот предназначен для запуска полноценного цикла обучения базовой модели (Model A) на датасете PTB-XL в среде Google Colab с использованием GPU. 

**Какие SOTA-подходы здесь реализованы:**
1. **Model Architecture**: Легкая ResNet-подобная архитектура `encoder_cnn` 
2. **Loss & Weights**: `BCEWithLogitsLoss` с учетом дисбаланса классов (`pos_weights`)
3. **Label Smoothing (0.1)**: Сглаживание меток для предотвращения переобучения
4. **WeightedRandomSampler**: Борьба с сильным дисбалансом классов (особенно HYP)
5. **Scheduler**: `Linear Warmup` (5 эпох) + `CosineAnnealingLR`
6. **Gradient Accumulation**: Эффективный размер батча = 128 (64 * 2)

Результаты, чекпоинты и логи синхронизируются **напрямую с Google Диском**, поэтому данные не потеряются при перезагрузке сессии.

In [2]:
# STEP 1: Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    print('Local run: Google Drive mount skipped')
    IN_COLAB = False


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# STEP 2: Find and enter the project root
import os
import sys
from pathlib import Path

PROJECT_DIR_CANDIDATES = [
    'ecg-diploma-main 3',
    'ecg-diploma-main',
    'ecg-diploma',
]

try:
    import google.colab
    IN_COLAB = True
except Exception:
    IN_COLAB = False


def looks_like_project(path: Path) -> bool:
    return (
        (path / 'src' / 'models' / 'model_baseline' / 'model_a_baseline.py').exists()
        and (path / 'data_preprocessed' / 'ptbxl_sota_100hz_diagnostic_superclass.npz').exists()
    )

candidates = []
if IN_COLAB:
    drive_root = Path('/content/drive/MyDrive')
    candidates += [drive_root / name for name in PROJECT_DIR_CANDIDATES]
    if drive_root.exists():
        candidates += [p for p in drive_root.iterdir() if p.is_dir() and p.name in PROJECT_DIR_CANDIDATES]
else:
    cwd = Path.cwd().resolve()
    candidates += [cwd, *cwd.parents]

PROJECT_ROOT = next((p for p in candidates if looks_like_project(p)), None)
if PROJECT_ROOT is None:
    checked = '\n'.join(str(p) for p in candidates[:20])
    raise FileNotFoundError(f'Project root was not found. Checked paths:\n{checked}')

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA exists:', (PROJECT_ROOT / 'data_preprocessed' / 'ptbxl_sota_100hz_diagnostic_superclass.npz').exists())
print('CWD:', Path.cwd())
!ls -la


PROJECT_ROOT: /content/drive/MyDrive/ecg-diploma-main 3
DATA exists: True
CWD: /content/drive/MyDrive/ecg-diploma-main 3
total 101
-rw------- 1 root root   254 May 11 16:31 CODEOWNERS
-rw------- 1 root root  1852 May 12 12:23 CONTRIBUTING.md
drwx------ 2 root root  4096 May 12 12:51 data_preprocessed
-rw------- 1 root root  6148 May 11 16:31 .DS_Store
-rw------- 1 root root 14266 May 12 17:57 eval_final.py
drwx------ 2 root root  4096 May 11 16:31 experiments
drwx------ 2 root root  4096 May 11 16:31 .git
drwx------ 2 root root  4096 May 11 16:31 .github
-rw------- 1 root root  4839 May 11 16:31 .gitignore
drwx------ 2 root root  4096 May 12 18:15 notebooks
drwx------ 2 root root  4096 May 12 17:57 __pycache__
-rw------- 1 root root  3295 May 12 12:21 README.md
-rw------- 1 root root  1087 May 12 12:21 requirements.txt
drwx------ 3 root root  4096 May 12 11:34 results
-rw------- 1 root root    37 May 11 16:31 results_summary.csv
-rw------- 1 root root  1457 May 12 12:23 SETUP.md
drwx--

In [3]:
# STEP 3: Install dependencies
# Run this cell only after a fresh Colab runtime reset.
try:
    import google.colab
    !pip install -q -r requirements.txt
    print('Dependencies checked')
except Exception:
    print('Local run: requirements install skipped')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 791.2 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 73.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 76.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
# STEP 4: Train Model A on GPU
from pathlib import Path

DATA_PATH = PROJECT_ROOT / 'data_preprocessed' / 'ptbxl_sota_100hz_diagnostic_superclass.npz'
OUT_DIR = PROJECT_ROOT / 'results' / 'model_a_baseline'
SCRIPT_PATH = PROJECT_ROOT / 'src' / 'models' / 'model_baseline' / 'model_a_baseline(1).py'
NUM_WORKERS = 2 if IN_COLAB else 0

assert DATA_PATH.exists(), f'DATA_PATH not found: {DATA_PATH}'
assert SCRIPT_PATH.exists(), f'SCRIPT_PATH not found: {SCRIPT_PATH}'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_PATH:', DATA_PATH)
print('OUT_DIR:', OUT_DIR)
print('SCRIPT_PATH:', SCRIPT_PATH)
print('NUM_WORKERS:', NUM_WORKERS)

!python "{SCRIPT_PATH}" \
    --data_path "{DATA_PATH}" \
    --output_dir "{OUT_DIR}" \
    --epochs 50 \
    --batch_size 64 \
    --accum_steps 2 \
    --warmup_epochs 5 \
    --label_smoothing 0.1 \
    --num_workers {NUM_WORKERS} \
    --use_weighted_sampler


DATA_PATH: /content/drive/MyDrive/ecg-diploma-main 3/data_preprocessed/ptbxl_sota_100hz_diagnostic_superclass.npz
OUT_DIR: /content/drive/MyDrive/ecg-diploma-main 3/results/model_a_baseline
SCRIPT_PATH: /content/drive/MyDrive/ecg-diploma-main 3/src/models/model_baseline/model_a_baseline(1).py
NUM_WORKERS: 2
Global seed set to 42

  PTB-XL SOTA Training -- MODEL_A_BASELINE
  Device:       cuda  |  AMP: ON
  Batch size:   64 x 2 accum = 128 effective
  LR schedule:  Warmup 5ep -> CosineAnnealing
  Label smooth: 0.1
  Epochs:       50  |  Patience: 10
  Output:       /content/drive/MyDrive/ecg-diploma-main 3/results/model_a_baseline

  Data path:    /content/drive/MyDrive/ecg-diploma-main 3/data_preprocessed/ptbxl_sota_100hz_diagnostic_superclass.npz
✅ Global seed set: 42
✅ Loaded train split: 17418 samples
   Signal shape: (12, 1000)
   Classes: ['NORM', 'MI', 'STTC', 'CD', 'HYP']
   Augmentation: ON
   Label distribution:
     NORM: 7596 (43.6%)
     MI: 4379 (25.1%)
     STTC: 4087 (23